In [3]:
import pandas as pd
import psycopg2, os, time

conn = psycopg2.connect(
  host="localhost", port=5432,
  dbname="insurance", user="insurance_user", password="root"
)   
df = pd.read_sql("SELECT * FROM claims LIMIT 1332", conn)
conn.close()
print(df.shape)

(528, 5)


/tmp/ipykernel_932203/1461773031.py:8: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("SELECT * FROM claims LIMIT 1332", conn)


In [4]:
csv_path     = "claims_sample.csv"
parquet_path = "claims_sample.parquet"

df.to_csv(csv_path, index=False)
df.to_parquet(parquet_path, index=False)

csv_bytes     = os.path.getsize(csv_path)
parquet_bytes = os.path.getsize(parquet_path)
print(f"CSV:     {csv_bytes:,} bytes")
print(f"Parquet: {parquet_bytes:,} bytes")
print(f"Ratio:   {csv_bytes / parquet_bytes:.1f}x")

CSV:     61,008 bytes
Parquet: 51,268 bytes
Ratio:   1.2x


In [5]:
%timeit pd.read_csv(csv_path, usecols=["id", "amount"])
%timeit pd.read_parquet(parquet_path, columns=["id", "amount"])

1.17 ms ± 49.6 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
1.57 ms ± 38.8 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [7]:
  import subprocess
  result = subprocess.run(
      ["mc", "cp", parquet_path, "local/lakehouse/bronze/claims_sample.parquet"],
      capture_output=True, text=True
  )
  print(result.stdout or result.stderr)

`/home/shakhriyar/insurance/notebooks/claims_sample.parquet` -> `local/lakehouse/bronze/claims_sample.parquet`



In [13]:
import pandas as pd

df_back = pd.read_parquet(
    "s3://lakehouse/bronze/claims_sample.parquet",
    columns=["id", "amount"],
    storage_options={
        "key": "minioadmin",
        "secret": "minioadmin123",
        "client_kwargs": {
            "endpoint_url": "http://localhost:9002"
        }
    }
)

print(df_back.shape)
df_back.head()

(528, 2)


,id,amount
0,216b2a5e-9ec4-474f-8472-5646b953fb9e,224.05
1,b09ce0bd-1a35-40ba-a8c0-93d12b8d98c3,12570.41
2,6cb35a75-660f-4e3f-9a17-c414e9f830f9,11622.41
3,6efae3f7-601d-49eb-ab84-6ea24081923a,11951.04
4,a06398de-982e-448d-8ed0-a52763f0b468,345.80


In [14]:
pip install pandas pyarrow psycopg2-binary s3fs

Note: you may need to restart the kernel to use updated packages.
